# Lezione 8 — QAOA e ottimizzazione combinatoria

## Obiettivi di apprendimento

- Tradurre un problema di ottimizzazione (Max-Cut) in un Hamiltoniano di costo
- Costruire e addestrare un circuito QAOA
- Campionare il circuito addestrato per ottenere soluzioni concrete

Molti problemi del mondo reale sono problemi di ottimizzazione combinatoria: distribuire percorsi di consegna, raggruppare clienti, progettare reti di telecomunicazioni. In questa lezione lavori con uno dei problemi di riferimento in questo campo, Max-Cut, e applichi l'Algoritmo Quantistico Approssimato di Ottimizzazione (QAOA) per risolverlo con un circuito addestrabile.

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum networkx

## Il problema Max-Cut

Max-Cut parte da un grafo: un insieme di nodi collegati da archi. Il tuo compito è dividere i nodi in due gruppi in modo che il numero di archi "tagliati" (con un nodo in ciascun gruppo) sia il massimo possibile. È un classico problema combinatorio e, con QAOA, lo affronti con un computer quantistico NISQ (rumoroso e senza correzione degli errori, come hai visto nella seconda parte della lezione 1).

## Tradurre il problema in un Hamiltoniano di costo

L'idea centrale di QAOA consiste nel tradurre il problema di ottimizzazione in un Hamiltoniano di costo: un operatore quantistico il cui valore di attesa, misurato su uno stato concreto, coincide con il numero di archi tagliati da quella divisione dei nodi.

Per Max-Cut, costruisci questo Hamiltoniano sommando un termine `Z_i · Z_j` per ogni arco tra i nodi i e j: questo termine vale -1 se i due nodi si trovano nello stesso gruppo, e +1 se si trovano in gruppi diversi. Più è alto il valore di attesa dell'Hamiltoniano, migliore è il taglio che rappresenta.

## Il circuito QAOA: alternare costo e miscelazione

Il circuito inizia mettendo tutti i qubit in sovrapposizione con porte Hadamard, rappresentando così tutte le divisioni possibili contemporaneamente. Poi alterna due blocchi, ciascuno con il proprio parametro addestrabile:

- L'Hamiltoniano di costo, parametrizzato da un angolo che chiami γ (gamma): spinge lo stato verso le divisioni che tagliano più archi
- L'Hamiltoniano di miscelazione, parametrizzato da un angolo che chiami β (beta): mescola lo stato in modo che il circuito esplori altre divisioni oltre a quella che sembra già buona, evitando di restare bloccato su una soluzione mediocre

Più volte ripeti questo ciclo costo-miscelazione (parametro `p` nella letteratura di QAOA), più il circuito può avvicinarsi alla soluzione ottimale, a costo di un circuito più profondo. Ricorda la seconda parte della lezione 1: più profondità significa più esposizione alla decoerenza.

## Dalla teoria a Cirq: `tfq.util.exponential`

Cirq e TFQ non ti obbligano a scomporre a mano ogni Hamiltoniano in porte quantistiche elementari. La funzione `tfq.util.exponential(operadores, coeficientes)` costruisce direttamente il circuito che applica `exp(-i · coeficiente · operatore)`, esattamente l'operazione che ti serve per applicare un Hamiltoniano per un tempo controllato dal parametro addestrabile.

## Leggere la risposta: campionare il circuito addestrato

Dopo aver addestrato i parametri γ e β, misuri il circuito molte volte con `tfq.layers.Sample`. Ogni misura ti dà una possibile divisione dei nodi in due gruppi; la divisione che si ripete più spesso (o quella con il taglio maggiore tra i campioni) è la tua soluzione approssimata al problema Max-Cut.

Di seguito trovi l'esempio completo commentato, con un grafo piccolo (un ciclo di sei nodi) in modo che l'addestramento sia rapido durante una sessione di lezione.

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Grafo ciclo de seis nodos: 0-1-2-3-4-5-0
grafo = nx.cycle_graph(6)
nx.draw_networkx(grafo)
plt.title("Grafo de partida: ciclo de seis nodos")
plt.show()

qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

# Estado inicial: todos los qubits en superposición (todas las divisiones a la vez)
estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))

In [ ]:
# Hamiltoniano de mezcla: suma de X sobre cada qubit
hamiltoniano_mezcla = sum(cirq.PauliString(cirq.X(qubits[i])) for i in range(len(qubits)))

# Hamiltoniano de coste: un término Z_i . Z_j por cada arista del grafo
hamiltoniano_coste = grafo.number_of_edges() / 2
for nodo_a, nodo_b in grafo.edges():
    hamiltoniano_coste += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))

In [ ]:
p = 1  # número de repeticiones del ciclo coste-mezcla
gamma, beta = sympy.symbols("gamma beta")

circuito_qaoa = cirq.Circuit()
circuito_qaoa += tfq.util.exponential(
    operators=[hamiltoniano_coste, hamiltoniano_mezcla],
    coefficients=[gamma, beta],
)

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)

modelo_qaoa = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

In [ ]:
# Entrenas para que el valor esperado del corte se acerque al máximo teórico (etiqueta ficticia 0)
historial = modelo_qaoa.fit(entradas, np.array([0]), epochs=200)

plt.plot(historial.history["loss"])
plt.title("QAOA — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

Con la perdita vicina a zero, i parametri γ e β sono ormai addestrati. Ora si tratta di campionare il circuito per leggere divisioni concrete e confrontarle con il taglio massimo reale, che su un grafo così piccolo puoi calcolare per forza bruta.

In [ ]:
parametros_entrenados = modelo_qaoa.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)
muestras = tfq.layers.Sample()(
    circuito_con_datos,
    symbol_names=[gamma, beta],
    symbol_values=parametros_entrenados,
    repetitions=1000,
)

cortes_obtenidos = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

corte_maximo = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in __import__("itertools").combinations(grafo.nodes(), r)
)

print("Corte medio obtenido:", np.mean(cortes_obtenidos))
print("Mejor corte posible (fuerza bruta):", corte_maximo)

Con un grafo così piccolo puoi calcolare il taglio massimo reale per forza bruta e confrontarlo: vedrai che il taglio medio ottenuto da QAOA si avvicina parecchio a quel massimo, anche se con una sola ripetizione (`p = 1`) quasi mai lo raggiunge in modo perfetto.

## Riepilogo dei punti chiave

- QAOA traduce un problema di ottimizzazione combinatoria (Max-Cut) in un Hamiltoniano di costo, il cui valore di attesa misura la qualità di una soluzione
- Il circuito alterna un blocco di costo (parametrizzato da γ) e un blocco di miscelazione (parametrizzato da β), ripetuti `p` volte
- `tfq.util.exponential` costruisce direttamente il circuito che applica un Hamiltoniano per un tempo dato da un parametro addestrabile
- Dopo l'addestramento, campioni il circuito molte volte per ottenere soluzioni concrete al problema
- Aumentare `p` (più ripetizioni del ciclo costo-miscelazione) di solito migliora la soluzione, a costo di un circuito più profondo